# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 1 — Collecting the Data

**Objective:** Retrieve SpaceX launch data, enrich it using the SpaceX API, clean the dataset, and export the Falcon 9 records to `dataset_part_1.csv`.

This notebook follows the supplied IBM lab structure and completes the missing code cells.

## 1. Import libraries and configure Pandas

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 2. Define helper functions

These functions use IDs in the launch records to retrieve rocket, launchpad, payload, and core details.

In [ ]:
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get(
                f"https://api.spacexdata.com/v4/rockets/{x}", timeout=30
            ).json()
            BoosterVersion.append(response.get('name'))


def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get(
                f"https://api.spacexdata.com/v4/launchpads/{x}", timeout=30
            ).json()
            Longitude.append(response.get('longitude'))
            Latitude.append(response.get('latitude'))
            LaunchSite.append(response.get('name'))


def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get(
                f"https://api.spacexdata.com/v4/payloads/{load}", timeout=30
            ).json()
            PayloadMass.append(response.get('mass_kg'))
            Orbit.append(response.get('orbit'))


def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get(
                f"https://api.spacexdata.com/v4/cores/{core['core']}", timeout=30
            ).json()
            Block.append(response.get('block'))
            ReusedCount.append(response.get('reuse_count'))
            Serial.append(response.get('serial'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)

        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## 3. Request and parse the SpaceX launch data

First, check the live API response. Then use the fixed JSON dataset supplied for this lab so results are consistent.

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url, timeout=30)

print("HTTP status code:", response.status_code)
print("Response preview:")
print(response.text[:500])

In [ ]:
static_json_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

response = requests.get(static_json_url, timeout=30)
response.raise_for_status()
launch_data = response.json()

data = pd.json_normalize(launch_data)
data.head()

## 4. Keep relevant columns and clean the launch records

In [ ]:
data = data[
    ['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']
].copy()

# Keep launches with exactly one core and one payload.
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Extract the single core and payload IDs from their lists.
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert UTC timestamp to a date.
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Restrict the dataset to launches on or before 13 November 2020.
data = data[data['date'] <= datetime.date(2020, 11, 13)].copy()

data.head()

## 5. Enrich the data using the SpaceX API

In [ ]:
# Lists used to collect API results
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

In [ ]:
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

print("Number of launch records:", len(data))
print("Booster versions retrieved:", len(BoosterVersion))
print("Launch sites retrieved:", len(LaunchSite))
print("Payload records retrieved:", len(PayloadMass))
print("Core records retrieved:", len(Outcome))

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

data = pd.DataFrame(launch_dict)
data.head()

In [ ]:
# Inspect the shape and summary of the collected dataset
print("Dataset shape:", data.shape)
data.info()
display(data.describe(include='all'))

## 6. Filter the dataframe to include only Falcon 9 launches

In [ ]:
data_falcon9 = data[data['BoosterVersion'] != 'Falcon 1'].copy()

# Reset FlightNumber to a consecutive sequence.
data_falcon9.loc[:, 'FlightNumber'] = range(1, data_falcon9.shape[0] + 1)

data_falcon9.head()

## 7. Data wrangling — handle missing values

In [ ]:
# Count missing values in each column.
data_falcon9.isnull().sum()

In [ ]:
# Calculate the mean payload mass, ignoring missing values.
payload_mass_mean = data_falcon9['PayloadMass'].mean()
print("Mean payload mass (kg):", payload_mass_mean)

# Replace missing payload masses with the mean.
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(
    np.nan, payload_mass_mean
)

# Verify missing-value counts. LandingPad may legitimately remain missing.
data_falcon9.isnull().sum()

## 8. Export the cleaned dataset

The resulting CSV can be used in the next lab.

In [ ]:
output_file = 'dataset_part_1.csv'
data_falcon9.to_csv(output_file, index=False)

print(f"Saved {len(data_falcon9)} Falcon 9 launch records to {output_file}")
display(data_falcon9.head())

## Completed

The notebook:
1. Requested SpaceX launch data.
2. Parsed and filtered launch records.
3. Retrieved additional rocket, payload, launchpad, and core information.
4. Created a structured Pandas DataFrame.
5. Kept Falcon 9 launches and handled missing payload masses.
6. Exported the cleaned dataset as `dataset_part_1.csv`.

**Note:** Run the notebook from top to bottom with an internet connection. The API enrichment cells make multiple requests and may take a little time.